# Ocean Modelling with Veros

### 🌊 The Sverdrup Relation 🌊 

In this exercise we investigate the [Sverdrup relation](https://en.wikipedia.org/wiki/Sverdrup_balance) (also known as Sverdrup balance). The Sverdrup relation is perhaps the most important balance in the ocean, because it connects *transport* (of major ocean currents) to *wind stress* (an atmospheric parameter). **It is proof that winds are the dominant driving factor for the large-scale ocean circulation.** 

As a formula, it reads:

\begin{equation}
V = \frac{1}{\rho_0 \beta} \nabla \times \tau
\end{equation}

Here, $V$ is the meridional (north-south) transport, $\rho_0 \approx 1024 ~ kg m^{-3}$ the density of sea water, $\beta$ the Rossby parameter, and $\tau$ the surface wind stress. 

Specifically, we study the connection of the Sverdrup relation to the barotropic streamfunction $\Psi$, for which

\begin{equation}
\frac{\partial}{\partial y} \Psi = -U, \quad \frac{\partial}{\partial x} \Psi = V
\end{equation}

$U$ and $V$ are defined as

\begin{equation}
U(x, y, t) = \int_{-D}^0 u(x, y, z, t) ~ \mathrm{d}z, \quad
V(x, y, t) = \int_{-D}^0 v(x, y, z, t) ~ \mathrm{d}z
\end{equation}

where $u$ is the velocity in zonal (east-west) direction, and $v$ the velocity in meridional (north-south) direction.

$\Psi$ is a handy tool to visualize the ocean circulation in 2 dimensions (without having to deal with $u$ and $v$ separately), because flow will always follow its contours.

## Step 1: Load output and inspect $\Psi$

With the model output ready, we can now begin to work with it. 

First, we need to import the packages that we will use in the exercise. You are likely already familiar with [NumPy](https://numpy.org/doc/stable/) and [matplotlib](https://matplotlib.org/stable/contents.html), which are Python packages that deal with mathematical operations and plotting, respectively. 

You might have not have used [xarray](http://xarray.pydata.org/en/stable/) yet. It is one of the most commonly used Python packages when dealing with multi-dimensional datasets. In climate modeling, we often have to deal with very complicated output files, and xarray makes dealing with them a lot easier.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

To open the generated output, you simply call `xr.open_dataset()`:

In [ ]:
ds = xr.open_dataset('acc_sverdrup.averages.nc')

In the cell below, you can inspect the model output. The dataset is divided into two compartments, **coordinates** and **variables**. 

The coordinates describe the grid *dimensions*: width, length, depth and time.

The variables show the physical *values* assigned to each grid cell, such as temperature, density, velocity, etc.

In [ ]:
ds

#### A quick guide to xarray
*(feel free to skip if you are already familiar with xarray or want to figure it out on your own)*

- `ds['var_name']` or `ds.var_name` allow you to inspect and work with specific variables from the set; 'var_name' could be for example 'temp' for temperature, 'u' for zonal speed, or 'salt' for salinity. 

- You can plot the output using `ds['var_name'].plot()`. A 1D array will be shown as a line plot, a 2D array as a heat map, and any higher dimensions as a histogram.

- You can select coordinates using `ds['var_name'].isel(dim_name=n)`, where *'dim_name'* is the dimension you want to select and *n* is the index. For example, if you'd like to see the 2D plot of sea surface temperature at the last timestep, you should use `ds['temp'].isel(Time=-1, zt=-1).plot()`

- It is also possible to perform array operations on the output variables, such as finding maxima, minima, sums or arithmetic means over specific dimensions, such as:
 - `ds['var_name'].mean(dim="Time")`
 - `ds['var_name'].sum(dim=("xu, yu"))`
 - `ds['var_name'].max(dim=("xu", "zw"))`
 
- xarray stores metadata about the variables alongside its values. If you wish to convert a specific variable to a plain NumPy array, use: `ds['var_name'].values`
 
- Most commands can be chained, for example: `ds['psi'].isel(xu=15).mean(dim="yu").plot()` shows the meridional mean of the barotropic stream function at the zonal position xu\[15\]=30°E for the entire runtime of the simulation. The plot will show the $\Psi$ values in units of \[m$^3$/s\] on the y-axis and the time in years on the x-axis.

In [ ]:
# Try out some of the commands above (or make up your own!) in this cell.
ds['psi']

Now use your knowledge of xarray and inspect $\Psi$ to see if the model run has reached a steady state.

(In the steady state, the barotropic stream function is approximately constant in time: $\partial \Psi / \partial t = 0$.)

**Plot the 1D curve of the maximum value of $\Psi$ over time and check whether the model is fully spun up, i.e., whether it is approximately constant.**

## Step 2: Compute wind stress curl

With all the technical model details in order, we can now get back to physics.

Before you can compute $V_{Sv}$, you must define some constants and compute the curl of the wind stress $\tau$.

In the cell below, the Rossby parameter $\beta$ is defined. When you attempt to run it, you will get an error, as some constants are missing. 

**Define $\rho_0$, $\Omega$, and the Earth's radius, $a$.**

Now you need to extract some variables from the input dataset. Namely, the output barotropic stream function (to compare with the computed $\Psi_{Sv}$), and the grid coordinates $x$ and $y$.

**Extract and define variables `psi`, `yt` and `xt`** as NumPy arrays from `ds`.

Lastly, you need the values for the grid spacings $dx$ and $dy$ in order to compute the gradients of windstress components and the integral $\int V ~ \mathrm{d}x$. The grid spacing $dx$ changes with latitude due to the Earth's sphericity. 

What are the units of `xt` and `yt`?

**Correct the definitions below to express $dx$ and $dy$ in meters.**

Write two functions can integrate and compute the gradient of an array. You can use fex. `np.nancumsum` for integration.

In [ ]:
def integrate():
    ...


def gradient():
    ...

**Now compute $\nabla \times \tau$.**

*Hint: The definition of the curl in 2D is $\nabla \times f(x,y) = \frac{\partial}{\partial x} f_y(x,y) - \frac{\partial}{\partial y} f_x(x,y)$*

**Check your solution by ploting $\nabla \times \tau$, using `imshow` or another 2D plotting command.**

Does the calculated $\nabla \times \tau$ make sense?

## Step 4: Compute $\Psi_\text{Sv}$ and compare

The task is now to compare the output barotropic streamfunction `ds['psi']` to $\Psi_{Sv}$ calculated from the Sverdrup relation:

\begin{equation}
\Psi_{Sv} = \int_{east}^{west}V_{Sv} ~ \mathrm{d}x
\end{equation}

*Note that the integral goes from east to west, as we define $\Psi(x_\text{east}) = 0$.*

**Use the definition of $V_{Sv}$ to compute it from the defined constants and the calculated curl of $\tau$. Then, compute the barotropic stream function $\Psi_{Sv}$ and compare it to the $\Psi$ from the model output by examining their respective plots.**

## Step 5: Conclusions

Congratulations, you're almost done! Let's take a moment to reflect on what we have learned doing all this technical work. To complete the exercise, answer the following:

**1. How does the diagnostic streamfunction differ from the one calculated from the Sverdrup balance?**

**2. How do these two solutions compare in the open ocean vs. at the boundaries?**

**3. How well does this idealized model reflect real winds and ocean currents?**